# Factor Polynomial Simulation

จำลองการทำงานของ `factor_polynomial` และ `try_factor_quadratic`  
จาก `src/algebra/polynomial/factor.hpp`

## อัลกอริธึม: factor_polynomial

```
factor_polynomial(poly):
1. สกัด monomial GCD  →  common_monomial (เช่น x จาก x²+x)
2. หาร poly ด้วย common_monomial
3. สกัด GCD ของสัมประสิทธิ์ทั้งหมด  →  numeric_factor
4. Normalize: leading coeff เป็นบวก (ถ้าลบ คูณ numeric_factor ด้วย -1)
5. ถ้า univariate + degree=2: try_factor_quadratic
6. มิฉะนั้น: irreducible factor
```

## อัลกอริธึม: try_factor_quadratic — Divisor Enumeration

สำหรับ a·x² + b·x + c:
```
1. ตรวจ: a, b, c ต้องเป็น integer ทั้งหมด
2. disc = b² - 4ac
3. ต้องการ disc ≥ 0 และเป็น perfect square
4. ถ้าผ่าน: sqrt_disc = √disc
5. หา factor pairs (p, q) ที่ p·q = a และ (r, s) ที่ r·s = c:
   for p in divisors(a):                ← O(√|a|)
       q = a / p
       for r in divisors(c):            ← O(√|c|)
           s = c / r
           # ตรวจว่า (px+r)(qx+s) = ax²+bx+c
           if p*s + q*r == b: found!
           if p*(-s) + q*(-r) == b: found! (with sign flip)
           ... (ลอง combinations ทั้ง 4)
```

In [ ]:
from __future__ import annotations
import math
from fractions import Fraction


# ─────────────────────────────────────────────
# Minimal Monomial + Polynomial representation
# (mirrors C++ Monomial and Polynomial classes)
# ─────────────────────────────────────────────

class Monomial:
    """Product of variables: x^a · y^b · ..."""

    def __init__(self, vars_: dict[str, int] | None = None):
        # ตัดตัวแปรที่ exponent = 0 ออก
        self.vars_ = {v: e for v, e in (vars_ or {}).items() if e != 0}

    def is_constant(self) -> bool:
        return len(self.vars_) == 0

    def total_degree(self) -> int:
        return sum(self.vars_.values())

    def degree_of(self, var: str) -> int:
        return self.vars_.get(var, 0)

    def variable_names(self) -> list[str]:
        return sorted(self.vars_.keys())

    def __mul__(self, other: "Monomial") -> "Monomial":
        result = dict(self.vars_)
        for v, e in other.vars_.items():
            result[v] = result.get(v, 0) + e
        return Monomial({v: e for v, e in result.items() if e != 0})

    def __truediv__(self, other: "Monomial") -> "Monomial":
        result = dict(self.vars_)
        for v, e in other.vars_.items():
            result[v] = result.get(v, 0) - e
        return Monomial({v: e for v, e in result.items() if e != 0})

    def divisible_by(self, other: "Monomial") -> bool:
        return all(self.vars_.get(v, 0) >= e for v, e in other.vars_.items())

    def __eq__(self, other) -> bool:
        return isinstance(other, Monomial) and self.vars_ == other.vars_

    def __hash__(self):
        return hash(tuple(sorted(self.vars_.items())))

    def __repr__(self) -> str:
        if not self.vars_:
            return "1"
        parts = []
        for v in sorted(self.vars_):
            e = self.vars_[v]
            parts.append(v if e == 1 else f"{v}^{e}")
        return "·".join(parts)


class Polynomial:
    """Sum of coefficient × monomial terms"""

    def __init__(self, terms: dict[Monomial, float] | None = None):
        self.terms: dict[Monomial, float] = {}
        for m, c in (terms or {}).items():
            if abs(c) > 1e-12:
                self.terms[m] = c

    @classmethod
    def from_univariate(cls, coeffs: list[float], var: str = "x") -> "Polynomial":
        """สร้างจาก list coefficients เรียงตาม degree สูงสุด"""
        n = len(coeffs) - 1
        terms = {}
        for i, c in enumerate(coeffs):
            deg = n - i
            m = Monomial({var: deg} if deg > 0 else {})
            if abs(c) > 1e-12:
                terms[m] = c
        return cls(terms)

    def is_zero(self) -> bool:
        return len(self.terms) == 0

    def degree(self) -> int:
        return max((m.total_degree() for m in self.terms), default=0)

    def variables(self) -> list[str]:
        vs = set()
        for m in self.terms:
            vs.update(m.variable_names())
        return sorted(vs)

    def is_univariate(self) -> bool:
        return len(self.variables()) <= 1

    def single_variable(self) -> str:
        vs = self.variables()
        return vs[0] if vs else "x"

    def coeff_of_degree(self, d: int) -> float:
        total = 0.0
        for m, c in self.terms.items():
            if m.total_degree() == d:
                total += c
        return total

    def monomial_gcd(self) -> Monomial:
        """GCD ของ monomials ทั้งหมด: min exponent ต่อตัวแปร"""
        if not self.terms:
            return Monomial()
        all_vars = set()
        for m in self.terms:
            all_vars.update(m.variable_names())
        result = {}
        for v in all_vars:
            min_exp = min(m.degree_of(v) for m in self.terms)
            if min_exp > 0:
                result[v] = min_exp
        return Monomial(result)

    def coefficient_gcd(self) -> float:
        """GCD ของ integer coefficients"""
        int_coeffs = [round(c) for c in self.terms.values()
                      if abs(c - round(c)) < 1e-9]
        if len(int_coeffs) != len(self.terms):
            return 1.0
        g = 0
        for x in int_coeffs:
            g = math.gcd(g, abs(x))
        return float(g) if g > 0 else 1.0

    def divide_by_monomial(self, m: Monomial) -> "Polynomial":
        result = {}
        for mono, c in self.terms.items():
            new_mono = mono / m
            result[new_mono] = c
        return Polynomial(result)

    def __repr__(self) -> str:
        if not self.terms:
            return "0"
        # เรียงตาม degree สูงสุดก่อน
        items = sorted(self.terms.items(), key=lambda t: -t[0].total_degree())
        parts = []
        for m, c in items:
            if m.is_constant():
                parts.append(f"{c:g}")
            elif abs(c - 1.0) < 1e-9:
                parts.append(str(m))
            elif abs(c + 1.0) < 1e-9:
                parts.append(f"-{m}")
            else:
                parts.append(f"{c:g}·{m}")
        return " + ".join(parts).replace(" + -", " - ")


print("Classes loaded. Quick test:")
p = Polynomial.from_univariate([1.0, -5.0, 6.0])   # x²-5x+6
print(f"  x²-5x+6 = {p}")
print(f"  degree = {p.degree()}, vars = {p.variables()}")

In [ ]:
def get_divisors(n: int, verbose: bool = False) -> list[int]:
    """
    หา divisors ทั้งหมด (ทั้ง positive และ negative) ของ |n|
    ใช้ loop √|n| เหมือน C++
    """
    n = abs(n)
    if n == 0:
        return [0]
    result = []
    i = 1
    while i * i <= n:
        if n % i == 0:
            result.append(i)
            if i != n // i:
                result.append(n // i)
        i += 1
    # เพิ่ม negative divisors
    all_divs = sorted(set(result + [-d for d in result]))
    if verbose:
        print(f"  divisors({n}) = {all_divs}")
    return all_divs


def try_factor_quadratic(
    poly: Polynomial,
    verbose: bool = True,
) -> tuple[bool, list[tuple["Polynomial", int]]]:
    """
    พยายาม factor univariate degree-2 polynomial เป็น linear factors
    เฉพาะกรณีที่ discriminant เป็น perfect square

    Returns
    -------
    (success, factors)
    factors: list of (linear_poly, exponent)
    """
    var = poly.single_variable()
    a = poly.coeff_of_degree(2)
    b = poly.coeff_of_degree(1)
    c = poly.coeff_of_degree(0)

    if verbose:
        print(f"\ntry_factor_quadratic({poly})")
        print(f"  a={a}, b={b}, c={c}  (variable='{var}')")

    # ตรวจว่า integer coefficients
    a_int, b_int, c_int = round(a), round(b), round(c)
    if abs(a - a_int) > 1e-9 or abs(b - b_int) > 1e-9 or abs(c - c_int) > 1e-9:
        if verbose:
            print("  → coefficients ไม่ใช่ integer → ไม่ factor")
        return False, []

    # ตรวจ discriminant
    disc = b_int * b_int - 4 * a_int * c_int
    if verbose:
        print(f"  disc = b²-4ac = {b_int}²-4·{a_int}·{c_int} = {disc}")

    if disc < 0:
        if verbose:
            print("  → disc < 0 → complex roots → ไม่ factor over ℝ")
        return False, []

    sqrt_disc = math.isqrt(disc)
    if sqrt_disc * sqrt_disc != disc:
        if verbose:
            print(f"  → disc={disc} ไม่ใช่ perfect square → ไม่ factor over ℤ")
        return False, []

    if verbose:
        print(f"  √disc = {sqrt_disc}  (perfect square ✓)")
        print(f"  \n  เริ่ม divisor enumeration สำหรับ a={a_int}, c={c_int}:")

    # Divisor enumeration: หา (p, q) = divisors of a  และ (r, s) = divisors of c
    a_divs = get_divisors(a_int, verbose=verbose)
    c_divs = get_divisors(c_int, verbose=verbose)

    factors = []
    found = False

    if verbose:
        print(f"\n  Loop: ลอง (p, r) ทุก pair — ตรวจ p·s + q·r = b={b_int}")
        print(f"  {'p':>4} {'q':>4} {'r':>4} {'s':>4} {'p·s+q·r':>10} {'match?':>8}")
        print(f"  {'─'*50}")

    for p in a_divs:
        if p == 0:
            continue
        q = a_int // p
        if a_int % p != 0:
            continue

        for r in c_divs:
            if r == 0:
                continue
            s = c_int // r
            if c_int % r != 0:
                continue

            # ลอง 4 sign combinations
            for sign_r, sign_s in [(1, 1), (1, -1), (-1, 1), (-1, -1)]:
                rr = r * sign_r
                ss = s * sign_s
                mid = p * ss + q * rr

                if verbose and abs(p) <= abs(a_int) and abs(rr) <= abs(c_int):
                    match = "✓ MATCH" if mid == b_int else ""
                    print(f"  {p:>4} {q:>4} {rr:>4} {ss:>4} {mid:>10}  {match}")

                if mid == b_int and not found:
                    # (p·x + r)(q·x + s) = a·x² + b·x + c
                    f1 = Polynomial.from_univariate([float(p), float(rr)], var)
                    f2 = Polynomial.from_univariate([float(q), float(ss)], var)
                    factors = [(f1, 1), (f2, 1)]
                    found = True
                    if verbose:
                        print(f"\n  FOUND: ({f1})·({f2})")

    if not found and verbose:
        print("\n  ไม่พบ factorization over ℤ")

    return found, factors

In [ ]:
def factor_polynomial(
    poly: Polynomial,
    verbose: bool = True,
) -> dict:
    """
    Main factorization pipeline:
    1. Monomial GCD
    2. Coefficient GCD
    3. Sign normalization
    4. Quadratic factorization (ถ้า univariate degree 2)
    """
    if verbose:
        print("=" * 60)
        print(f"factor_polynomial({poly})")
        print("=" * 60)

    result = {
        "numeric_factor": 1.0,
        "common_monomial": Monomial(),
        "factors": [],       # list of (Polynomial, exponent)
    }

    remaining = poly

    # ── Step 1: Monomial GCD ─────────────────────────────────────
    m_gcd = poly.monomial_gcd()
    if verbose:
        print(f"\n  Step 1: monomial_gcd = {m_gcd}")

    if not m_gcd.is_constant():
        result["common_monomial"] = m_gcd
        remaining = poly.divide_by_monomial(m_gcd)
        if verbose:
            print(f"    หลัง ÷ {m_gcd}: {remaining}")
    else:
        if verbose:
            print(f"    ไม่มี common monomial")

    # ── Step 2: Coefficient GCD ──────────────────────────────────
    c_gcd = remaining.coefficient_gcd()
    if verbose:
        print(f"\n  Step 2: coefficient_gcd = {c_gcd}")

    if abs(c_gcd - 1.0) > 1e-9 and c_gcd > 0:
        result["numeric_factor"] *= c_gcd
        # หาร terms ด้วย c_gcd
        remaining = Polynomial({m: v / c_gcd for m, v in remaining.terms.items()})
        if verbose:
            print(f"    numeric_factor = {c_gcd}")
            print(f"    หลัง ÷ {c_gcd}: {remaining}")
    else:
        if verbose:
            print(f"    ไม่มี common coefficient factor")

    # ── Step 3: Sign normalization ───────────────────────────────
    # leading term ควรเป็น positive
    if remaining.terms:
        # leading coeff = coeff of highest degree term
        highest = max(remaining.terms.keys(), key=lambda m: m.total_degree())
        lead = remaining.terms[highest]
        if verbose:
            print(f"\n  Step 3: leading coeff = {lead}")
        if lead < 0:
            result["numeric_factor"] *= -1
            remaining = Polynomial({m: -v for m, v in remaining.terms.items()})
            if verbose:
                print(f"    flip sign: numeric_factor *= -1 = {result['numeric_factor']}")
                print(f"    remaining = {remaining}")
        else:
            if verbose:
                print(f"    leading coeff > 0, ไม่ต้อง flip")

    # ── Step 4: Quadratic factorization ─────────────────────────
    if verbose:
        print(f"\n  Step 4: univariate={remaining.is_univariate()}, degree={remaining.degree()}")

    if remaining.is_univariate() and remaining.degree() == 2:
        ok, quad_factors = try_factor_quadratic(remaining, verbose=verbose)
        if ok:
            result["factors"] = quad_factors
        else:
            result["factors"] = [(remaining, 1)]
    else:
        result["factors"] = [(remaining, 1)]
        if verbose:
            print(f"    → irreducible factor: {remaining}")

    # ── Print result ─────────────────────────────────────────────
    if verbose:
        print(f"\n{'─'*60}")
        print(f"RESULT:")
        print(f"  numeric_factor   = {result['numeric_factor']}")
        print(f"  common_monomial  = {result['common_monomial']}")
        print(f"  factors:")
        for f, exp in result["factors"]:
            print(f"    ({f})^{exp}")
        # สร้าง string สวยงาม
        parts = []
        if abs(result["numeric_factor"] - 1.0) > 1e-9 or not result["common_monomial"].is_constant():
            nf = result["numeric_factor"]
            cm = result["common_monomial"]
            if not cm.is_constant():
                parts.append(f"{nf:g}·{cm}")
            else:
                parts.append(f"{nf:g}")
        for f, exp in result["factors"]:
            s = f"({f})"
            parts.append(s if exp == 1 else f"{s}^{exp}")
        print(f"\n  = {' · '.join(parts) if parts else str(poly)}")

    return result

## Test Case 1: x² - 5x + 6 = (x-2)(x-3)

Perfect square discriminant: disc = 25-24 = 1 → √1 = 1

In [ ]:
p1 = Polynomial.from_univariate([1.0, -5.0, 6.0])
r1 = factor_polynomial(p1)

assert len(r1["factors"]) == 2
print("\n✓ Test 1 passed: x²-5x+6 = (x-2)(x-3)")

## Test Case 2: 2x² + 8x + 6 = 2(x+1)(x+3)

มี numeric_factor = 2 ก่อน factor quadratic

In [ ]:
p2 = Polynomial.from_univariate([2.0, 8.0, 6.0])
r2 = factor_polynomial(p2)

assert abs(r2["numeric_factor"] - 2.0) < 1e-9
assert len(r2["factors"]) == 2
print("\n✓ Test 2 passed: 2x²+8x+6 = 2·(x+1)·(x+3)")

## Test Case 3: 3x³ + 6x² = 3x²(x+2)

มี common monomial = x²

In [ ]:
# 3x³ + 6x² — manually create terms
m_x3 = Monomial({"x": 3})
m_x2 = Monomial({"x": 2})
p3 = Polynomial({m_x3: 3.0, m_x2: 6.0})
print(f"polynomial: {p3}")

r3 = factor_polynomial(p3)

assert str(r3["common_monomial"]) == "x^2"
assert abs(r3["numeric_factor"] - 3.0) < 1e-9
print("\n✓ Test 3 passed: 3x³+6x² = 3·x²·(x+2)")

## Test Case 4: x² + 1  (ไม่ factor ได้ over ℝ)

disc = 0 - 4 = -4 < 0 → complex roots → irreducible

In [ ]:
p4 = Polynomial.from_univariate([1.0, 0.0, 1.0])  # x² + 1
r4 = factor_polynomial(p4)

assert len(r4["factors"]) == 1, "x²+1 ควรเป็น irreducible"
print("\n✓ Test 4 passed: x²+1 = irreducible (complex roots)")

## Test Case 5: x² - 2  (disc=8 ไม่ใช่ perfect square)

roots คือ ±√2 → irrational → ไม่ factor over ℤ

In [ ]:
p5 = Polynomial.from_univariate([1.0, 0.0, -2.0])  # x² - 2
r5 = factor_polynomial(p5)

assert len(r5["factors"]) == 1, "x²-2 ควร irreducible over ℤ"
print("\n✓ Test 5 passed: x²-2 = irreducible over ℤ (√2 irrational)")

## Test Case 6: -6x² + x + 2 — leading coefficient ลบ

ต้อง normalize: คูณ numeric_factor ด้วย -1 ก่อน factor  
= -1·(6x² - x - 2) = -1·(2x+1)(3x-2)

In [ ]:
p6 = Polynomial.from_univariate([-6.0, 1.0, 2.0])  # -6x² + x + 2
r6 = factor_polynomial(p6)

assert r6["numeric_factor"] < 0, "numeric_factor ควรเป็นลบ"
print(f"\n✓ Test 6 passed: numeric_factor={r6['numeric_factor']}")

## สรุป Loop Structure

```
factor_polynomial(poly):
│
├── monomial_gcd():                           ← O(n·v) — n terms, v vars
│       for each var: min exponent across all terms
│
├── coefficient_gcd():                         ← O(n) — Euclidean GCD of n coeffs
│
├── sign normalization:                         ← O(n) — find + flip leading coeff
│
└── try_factor_quadratic(poly):               ← เฉพาะ degree=2 univariate
    ├── check disc = b²-4ac
    ├── check perfect square: isqrt(disc)²=disc
    ├── for p in divisors(a):                 ← O(√|a|)
    │       for r in divisors(c):             ← O(√|c|)
    │           for sign in {+1,-1}²:         ← 4 combinations
    │               if p·s + q·r == b: FOUND
    Total: O(√|a| · √|c|)
```

| Test | Polynomial | Step ที่น่าสนใจ | ผลลัพธ์ |
|------|-----------|----------------|--------|
| 1 | x²-5x+6 | disc=1 (perfect sq) | (x-2)(x-3) |
| 2 | 2x²+8x+6 | coeff_gcd=2 | 2(x+1)(x+3) |
| 3 | 3x³+6x² | monomial_gcd=x² | 3·x²·(x+2) |
| 4 | x²+1 | disc=-4<0 | irreducible |
| 5 | x²-2 | disc=8 not perfect sq | irreducible over ℤ |
| 6 | -6x²+x+2 | flip leading sign | -1·(2x+1)(3x-2) |